# Ecommify — Unidad 4: Implementación de Particionamiento Declarativo

**Curso:** Maestría en Arquitectura de Software  
**Proyecto:** Ecommify — Plataforma de e-commerce  
**Actividad:** Etapa 2 — Implementación. Particionamiento declarativo tabla orders  

---

## Objetivo

Implementar particionamiento declarativo RANGE por `purchase_timestamp` en la tabla
orders, migrar los datos existentes y comparar el rendimiento de consultas
con y sin particionamiento mediante métricas cuantificables.

## Estrategia

- **Tipo:** RANGE por `purchase_timestamp`
- **Granularidad:** Anual con esquema hot/cold
- **Particiones:** 2016-2017 (COLD), 2018 (WARM), 2026 (HOT), DEFAULT
- **Tabla original:** orders (~99.441 registros)

---
## Paso 1 — Conexión a Supabase

In [1]:
!pip install psycopg2-binary --quiet

import psycopg2
from google.colab import userdata

SUPABASE_URI = userdata.get('SUPABASE_URI')
conn = psycopg2.connect(SUPABASE_URI)
conn.autocommit = True
cur = conn.cursor()
print('✅ Conexión a Supabase establecida correctamente.')

cur.execute('SELECT COUNT(*) FROM orders')
print(f'   orders actuales: {cur.fetchone()[0]:,} registros')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 12.5 MB/s eta 0:00:00
✅ Conexión a Supabase establecida correctamente.
   orders actuales: 99,441 registros


---
## Paso 2 — Distribución temporal de orders (línea base)

In [2]:
print('📊 Distribución de orders por año:')
cur.execute("""
    SELECT
        EXTRACT(YEAR FROM purchase_timestamp)::INT AS anio,
        COUNT(*) AS total
    FROM orders
    GROUP BY anio
    ORDER BY anio
""")
rows = cur.fetchall()
for anio, total in rows:
    print(f'   {anio}: {total:,} órdenes')

📊 Distribución de orders por año:
   2016: 329 órdenes
   2017: 45,101 órdenes
   2018: 54,011 órdenes


---
## Paso 3 — Benchmark ANTES del particionamiento

Se mide el tiempo de Q13 sobre la tabla original para comparar después.

In [3]:
import time

Q13 = """
SELECT
    EXTRACT(YEAR FROM purchase_timestamp)::INT AS anio,
    COUNT(*) AS total
FROM orders
WHERE purchase_timestamp >= '2018-01-01'
  AND purchase_timestamp <  '2019-01-01'
GROUP BY anio
"""

print('⏱️  Benchmark ANTES del particionamiento (tabla orders original):')
tiempos = []
for i in range(5):
    inicio = time.time()
    cur.execute(Q13)
    cur.fetchall()
    fin = time.time()
    tiempos.append((fin - inicio) * 1000)

avg_antes = sum(tiempos) / len(tiempos)
print(f'   Ejecuciones: {[f"{t:.1f}ms" for t in tiempos]}')
print(f'   Promedio ANTES: {avg_antes:.1f} ms')

# EXPLAIN
cur.execute(f'EXPLAIN (ANALYZE, BUFFERS) {Q13}')
plan = '\n'.join([r[0] for r in cur.fetchall()])
print(f'\n📋 Plan ANTES:')
print(plan)

⏱️  Benchmark ANTES del particionamiento (tabla orders original):
   Ejecuciones: ['210.2ms', '209.3ms', '209.0ms', '208.9ms', '209.9ms']
   Promedio ANTES: 209.5 ms

📋 Plan ANTES:
HashAggregate  (cost=3326.44..4555.28 rows=53814 width=12) (actual time=33.271..33.304 rows=1 loops=1)
  Group Key: (EXTRACT(year FROM purchase_timestamp))::integer
  Batches: 1  Memory Usage: 1561kB
  Buffers: shared hit=150
  ->  Index Only Scan using idx_orders_purchase on orders  (cost=0.29..1513.42 rows=53969 width=4) (actual time=0.034..23.843 rows=54011 loops=1)
        Index Cond: ((purchase_timestamp >= '2018-01-01 00:00:00+00'::timestamp with time zone) AND (purchase_timestamp < '2019-01-01 00:00:00+00'::timestamp with time zone))
        Heap Fetches: 0
        Buffers: shared hit=150
Planning:
  Buffers: shared hit=3
Planning Time: 0.164 ms
Execution Time: 33.886 ms


---
## Paso 4 — Crear tabla particionada

In [4]:
print('Creando tabla particionada orders_partitioned...')

cur.execute('DROP TABLE IF EXISTS orders_partitioned CASCADE')

cur.execute("""
CREATE TABLE orders_partitioned (
    id                      UUID NOT NULL,
    customer_id             UUID NOT NULL,
    order_status            VARCHAR(30) NOT NULL,
    purchase_timestamp      TIMESTAMPTZ NOT NULL,
    approved_at             TIMESTAMPTZ,
    delivered_carrier_date  TIMESTAMPTZ,
    delivered_customer_date TIMESTAMPTZ,
    estimated_delivery_date TIMESTAMPTZ,
    update_at               TIMESTAMPTZ NOT NULL DEFAULT NOW()
) PARTITION BY RANGE (purchase_timestamp)
""")
print('✅ Tabla padre creada.')

# Partición COLD: datos históricos 2016-2017
cur.execute("""
CREATE TABLE orders_2016_2017
    PARTITION OF orders_partitioned
    FOR VALUES FROM ('2016-01-01') TO ('2018-01-01')
""")
print('✅ Partición orders_2016_2017 (COLD) creada.')

# Partición WARM: 2018
cur.execute("""
CREATE TABLE orders_2018
    PARTITION OF orders_partitioned
    FOR VALUES FROM ('2018-01-01') TO ('2019-01-01')
""")
print('✅ Partición orders_2018 (WARM) creada.')

# Partición HOT: año actual
cur.execute("""
CREATE TABLE orders_2026
    PARTITION OF orders_partitioned
    FOR VALUES FROM ('2026-01-01') TO ('2027-01-01')
""")
print('✅ Partición orders_2026 (HOT) creada.')

# Partición DEFAULT
cur.execute("""
CREATE TABLE orders_default
    PARTITION OF orders_partitioned DEFAULT
""")
print('✅ Partición orders_default creada.')

Creando tabla particionada orders_partitioned...
✅ Tabla padre creada.
✅ Partición orders_2016_2017 (COLD) creada.
✅ Partición orders_2018 (WARM) creada.
✅ Partición orders_2026 (HOT) creada.
✅ Partición orders_default creada.


---
## Paso 5 — Migrar datos de orders → orders_partitioned

In [5]:
import time
print('Migrando datos...')
inicio = time.time()

cur.execute("""
INSERT INTO orders_partitioned
SELECT
    id, customer_id, order_status,
    purchase_timestamp, approved_at,
    delivered_carrier_date, delivered_customer_date,
    estimated_delivery_date, update_at
FROM orders
""")

fin = time.time()
print(f'✅ Migración completada en {(fin-inicio):.1f} segundos.')

# Verificar conteos por partición
print('\n📊 Registros por partición:')
for tabla in ['orders_2016_2017','orders_2018','orders_2026','orders_default']:
    cur.execute(f'SELECT COUNT(*) FROM {tabla}')
    print(f'   {tabla:<25} {cur.fetchone()[0]:>10,} registros')

cur.execute('SELECT COUNT(*) FROM orders_partitioned')
print(f'\n   TOTAL orders_partitioned: {cur.fetchone()[0]:,} registros')

Migrando datos...
✅ Migración completada en 1.1 segundos.

📊 Registros por partición:
   orders_2016_2017              45,430 registros
   orders_2018                   54,011 registros
   orders_2026                        0 registros
   orders_default                     0 registros

   TOTAL orders_partitioned: 99,441 registros


---
## Paso 6 — Crear índices por partición activa

In [6]:
print('Creando índices en particiones...')

cur.execute('CREATE INDEX idx_op_2018_customer ON orders_2018(customer_id)')
cur.execute('CREATE INDEX idx_op_2018_status   ON orders_2018(order_status)')
cur.execute('CREATE INDEX idx_op_2018_ts       ON orders_2018(purchase_timestamp)')
print('✅ Índices en orders_2018 creados.')

cur.execute('CREATE INDEX idx_op_cold_customer ON orders_2016_2017(customer_id)')
print('✅ Índice en orders_2016_2017 creado.')

print('\n✅ Todos los índices creados correctamente.')

Creando índices en particiones...
✅ Índices en orders_2018 creados.
✅ Índice en orders_2016_2017 creado.

✅ Todos los índices creados correctamente.


---
## Paso 7 — Benchmark DESPUÉS del particionamiento

La misma consulta con filtro de año sobre orders_partitioned — el partition pruning debería aislar solo orders_2018.

In [7]:
Q13_particionada = """
SELECT
    EXTRACT(YEAR FROM purchase_timestamp)::INT AS anio,
    COUNT(*) AS total
FROM orders_partitioned
WHERE purchase_timestamp >= '2018-01-01'
  AND purchase_timestamp <  '2019-01-01'
GROUP BY anio
"""

print('⏱️  Benchmark DESPUÉS del particionamiento (orders_partitioned):')
tiempos_despues = []
for i in range(5):
    inicio = time.time()
    cur.execute(Q13_particionada)
    cur.fetchall()
    fin = time.time()
    tiempos_despues.append((fin - inicio) * 1000)

avg_despues = sum(tiempos_despues) / len(tiempos_despues)
print(f'   Ejecuciones: {[f"{t:.1f}ms" for t in tiempos_despues]}')
print(f'   Promedio DESPUÉS: {avg_despues:.1f} ms')

# EXPLAIN
cur.execute(f'EXPLAIN (ANALYZE, BUFFERS) {Q13_particionada}')
plan = '\n'.join([r[0] for r in cur.fetchall()])
print(f'\n📋 Plan DESPUÉS:')
print(plan)

⏱️  Benchmark DESPUÉS del particionamiento (orders_partitioned):
   Ejecuciones: ['223.7ms', '214.1ms', '215.9ms', '214.2ms', '213.8ms']
   Promedio DESPUÉS: 216.3 ms

📋 Plan DESPUÉS:
HashAggregate  (cost=254.83..257.83 rows=200 width=12) (actual time=34.272..34.274 rows=1 loops=1)
  Group Key: (EXTRACT(year FROM orders_partitioned.purchase_timestamp))::integer
  Batches: 1  Memory Usage: 40kB
  Buffers: shared hit=975
  ->  Bitmap Heap Scan on orders_2018 orders_partitioned  (cost=4.16..253.48 rows=270 width=4) (actual time=3.305..25.698 rows=54011 loops=1)
        Recheck Cond: ((purchase_timestamp >= '2018-01-01 00:00:00+00'::timestamp with time zone) AND (purchase_timestamp < '2019-01-01 00:00:00+00'::timestamp with time zone))
        Heap Blocks: exact=826
        Buffers: shared hit=975
        ->  Bitmap Index Scan on idx_op_2018_ts  (cost=0.00..4.09 rows=270 width=0) (actual time=3.199..3.200 rows=54011 loops=1)
              Index Cond: ((purchase_timestamp >= '2018-01-01 00:

---
## Paso 8 — Resumen comparativo

In [8]:
mejora = ((avg_antes - avg_despues) / avg_antes) * 100

print('='*60)
print(' RESUMEN COMPARATIVO — Particionamiento orders')
print('='*60)
print(f'  Tiempo promedio SIN partición:  {avg_antes:.1f} ms')
print(f'  Tiempo promedio CON partición:  {avg_despues:.1f} ms')
print(f'  Reducción de tiempo:            {mejora:.1f}%')
print()
print('  Particiones creadas:')
print('    orders_2016_2017  COLD  → datos históricos Olist')
print('    orders_2018       WARM  → último año del dataset')
print('    orders_2026       HOT   → año en curso')
print('    orders_default    DEFAULT → captura fuera de rango')
print()
print('  Partition pruning activo: la consulta filtrada por')
print('  purchase_timestamp 2018 solo escanea orders_2018,')
print('  ignorando el resto de particiones.')
print('='*60)

cur.close()
conn.close()
print('\n✅ Análisis completo. Conexión cerrada.')

 RESUMEN COMPARATIVO — Particionamiento orders
  Tiempo promedio SIN partición:  209.5 ms
  Tiempo promedio CON partición:  216.3 ms
  Reducción de tiempo:            -3.3%

  Particiones creadas:
    orders_2016_2017  COLD  → datos históricos Olist
    orders_2018       WARM  → último año del dataset
    orders_2026       HOT   → año en curso
    orders_default    DEFAULT → captura fuera de rango

  Partition pruning activo: la consulta filtrada por
  purchase_timestamp 2018 solo escanea orders_2018,
  ignorando el resto de particiones.

✅ Análisis completo. Conexión cerrada.
